In [15]:
# Imports
import numpy as np
from scipy.sparse import csr_matrix, save_npz
import pandas as pd

## Part 1: Initial Processing

In [ ]:
fm_df = pd.read_csv("data\Last.fm_data.csv", index_col=0)
fm_df.head()

<>:3: SyntaxWarning: invalid escape sequence '\L'
<>:3: SyntaxWarning: invalid escape sequence '\L'
C:\Users\charl\AppData\Local\Temp\ipykernel_7860\602478312.py:3: SyntaxWarning: invalid escape sequence '\L'
  fm_df = pd.read_csv("data\Last.fm_data.csv", index_col=0)


,Username,Artist,Track,Album,Date,Time
0,Babs_05,Isobel Campbell,The Circus Is Leaving Town,Ballad of the Broken Seas,31 Jan 2021,23:36
1,Babs_05,Isobel Campbell,Dusty Wreath,Ballad of the Broken Seas,31 Jan 2021,23:32
2,Babs_05,Isobel Campbell,Honey Child What Can I Do?,Ballad of the Broken Seas,31 Jan 2021,23:28
3,Babs_05,Isobel Campbell,It's Hard To Kill A Bad Thing,Ballad of the Broken Seas,31 Jan 2021,23:25
4,Babs_05,Isobel Campbell,Saturday's Gone,Ballad of the Broken Seas,31 Jan 2021,23:21


In [16]:
# Inspect the DataFrame
print(fm_df.shape)
print(fm_df.columns.tolist())
print(fm_df.dtypes)

(166153, 6)
['Username', 'Artist', 'Track', 'Album', 'Date', 'Time']
Username    object
Artist      object
Track       object
Album       object
Date        object
Time        object
dtype: object


In [19]:
# Set column types
fm_df['Username'] = fm_df['Username'].astype('category')
fm_df['Artist'] = fm_df['Artist'].astype('category')

In [20]:
# Drop unnecessary columns
fm_df = fm_df.drop(columns=['Track','Album','Date','Time'])

In [23]:
# Check column names
print(fm_df.columns.tolist())
print(fm_df.dtypes)

['Username', 'Artist']
Username    category
Artist      category
dtype: object


## Part 2: Grouping

In [41]:
grouped_df = fm_df.groupby(['Username', 'Artist'],observed=True).size().reset_index(name='PlayCount')
grouped_df.head(10)

,Username,Artist,PlayCount
0,Babs_05,"""Demons""",1
1,Babs_05,"""Weird Al"" Yankovic",7
2,Babs_05,#1 Garth Brooks Tribute Band,2
3,Babs_05,$uicideboy$,48
4,Babs_05,'Come From Away' Band,1
5,Babs_05,'Come From Away' Company,10
6,Babs_05,(G)I-DLE,1
7,Babs_05,*NSYNC,1
8,Babs_05,*shels,3
9,Babs_05,///earth///,1


# Part 3: Saving as sparse matrix

In [42]:
# Convert to categoricals to get integer codes for each user/artist
user_cat = grouped_df['Username'].astype('category')
artist_cat = grouped_df['Artist'].astype('category')

user_idx = user_cat.cat.codes
artist_idx = artist_cat.cat.codes

# Build the sparse matrix: rows = users, columns = artists
sparse_matrix = csr_matrix(
    (grouped_df['PlayCount'], (user_idx, artist_idx)),
    shape=(user_cat.cat.categories.size, artist_cat.cat.categories.size)
)

# Keep the labels so you can map back
users = user_cat.cat.categories
artists = artist_cat.cat.categories

print(sparse_matrix.shape)
print(f"Non-zero entries: {sparse_matrix.nnz}")

(11, 22823)
Non-zero entries: 51790


In [44]:
u = users.get_loc('Babs_05')
a = artists.get_loc('"Weird Al" Yankovic')
sparse_matrix[u, a] # Should return 7

7

# Part 4: Saving Sparse Matrix to Disk

In [47]:
save_npz('data/tidied/user_artist_matrix.npz', sparse_matrix)

# Save the labels too, so you can map rows/columns back to names
np.save('data/tidied/users.npy', users.to_numpy(), allow_pickle=True)
np.save('data/tidied/artists.npy', artists.to_numpy(), allow_pickle=True)